## 4. Pseudo-Labeling on Generated Data

You must complete the previous setup step [0-setup-cuda128.ipynb](./0-setup-cuda128.ipynb), training step [1-training.ipynb](./1-training.ipynb) and generation step [3-generation.ipynb](./3-generation.ipynb).

In this section, we will perform the pseudo-labeling process for the generated data. Pseudo-labeling is used to refine and organize the data into the structured formats required by downstream tasks, especially for the TAO toolkit. It also provides the SAM2-based mask refinement and the Cosmos-Reason 1 7B-based captioner to generate captions for the generated images.

**Important**:
When enabling mask refinement, you must first finetune the SAM2 model to obtain the finetuned model weights. We will guide you through this process below.

**Why Mask Refinement:**
Our generation pipeline is not perfect, meaning the generated anomaly images may not fully align with the input masks. Mask refinement is utilized to solve this problem by employing a fine-tuned SAM2 model to refine the masks, thereby providing more precise labeling.

### 4.0 Setting Up the Environment

This notebook requires the user to set the environment variable `LOCAL_PROJECT_DIR` to the path of the PAIDF AnomalyGen repo. Remember to replace `FIXME` placeholder below with the correct path.

In [ ]:
# Set `LOCAL_PROJECT_DIR` for PAIDF AnomalyGen.
LOCAL_PROJECT_DIR="FIXME"
# Set the working directory to this path for the shell.
%cd {LOCAL_PROJECT_DIR}
# Use `cd ${LOCAL_PROJECT_DIR}` if you are copy-pasting this into a terminal.

import os

os.environ["LD_LIBRARY_PATH"] = (
    f"{LOCAL_PROJECT_DIR}/anaconda3/envs/cosmos-predict2/lib/python3.12/site-packages/nvidia/cudnn/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)
os.environ.pop("MPLBACKEND", None)

### (Optional) 4.1 SAM2 Finetuning

In this section, we will use the same training data that is used to train the AnomalyGen model in [1-training.ipynb](./1-training.ipynb) to finetune the SAM2 model. If you are not utilizing mask refinement, you can skip this section.

This finetuning process is effective with limited data because it uses the knowledge distillation technique. In general, finetuning the model with 100 image-mask pairs takes less than 30 minutes on a single GPU.

During the validation process, it will randomly enlarge the GT masks and then computes the mIoU for GT vs. enlarged masks and GT vs. refined masks. If the finetuned model is good, the mIoU from the refined masks should be higher than that from the randomly enlarged masks.

**Important**:
When the total size of the dataset is less than 100 samples, the entire dataset will be used as the validation set. This is because the validation set is used to select the best model weights and must be large enough to ensure enough data diversity.

Available arguments for the finetuning script are:

- `image_dir`: The directory containing the anomaly images.
- `mask_dir`: The directory containing the masks corresponding to the anomaly images.
- `output_dir`: The directory where the finetuned weights will be saved.
- `train_val_ratio`: Ratio for splitting the dataset into training and validation sets. Default is `0.8` which means 80% of the data is used for training and 20% for validation. When the total size of the dataset is less than 100 samples, the entire dataset will be used as the validation set.
- `epochs`: The training epochs. Default is `20`.
- `patient_epochs`: When the validation metric (mIoU) does not improve for this number of epochs, the training will be stopped. Default is `5`.
- `batch_size`: The batch size for training. Default is `4`.
- `lr`: The learning rate. Default is `2e-4`.
- `weight_decay`: The weight decay. Default is `1e-4`.
- `eval_dilate_sizes`: The candidate sizes for dilation during evaluation. Default is [7, 9, 11, 13].
- `eval_dbscan_eps`: DBSCAN eps parameter for clustering masks during evaluation. Default is 0.2.
- `eval_dbscan_min_samples`: DBSCAN min_sample parameter for clustering masks during evaluation. Default is 5.
- `eval_crop_ratio`: The expand ratio for the ROI during evaluation. Default is 2.0.
- `eval_strength`: The strength for the mask prompt during evaluation. Set to 0.0 to disable this feature. Default is 1.0.
- `eval_fallback_ratio`: The threshold ratio of the area for the refined mask that needs to be restored to the original one during evaluation. 0.5 means that if the area of the refined mask is smaller than the area of the input mask by 50%, the refined mask will be replaced with the input mask. Use 0.0 to disable this mechanism. Default is 0.5.

<font color="red">**Important**: This is a quick example, so the finetuned model may be suboptimal.
To improve quality, increase the number of training epochs to the default `20` (`--epochs 20`).</font>

In [ ]:
!conda run -n cosmos-predict2 \
    bash -c "python3 -m scripts.anomaly_gen.finetune_sam \
        --image_dir=datasets/MIIC/train_downsample/SEM_IC/anomaly_image \
        --mask_dir=datasets/MIIC/train_downsample/SEM_IC/mask \
        --output_dir=results/MIIC/sam2_finetuning \
        --epochs 1"

When using MIIC as the dataset, you should get results similar to the following metrics during the finetuning process:

```shell
Dilated masks metrics: {'miou': 0.880367636680603, 'mf1': 0.9363781809806824, 'mprecision': 0.880367636680603, 'mrecall': 1.0}
Refined masks metrics: {'miou': 0.9322577118873596, 'mf1': 0.9649413824081421, 'mprecision': 0.947699248790741, 'mrecall': 0.9828224778175354}
```

When the finetuning process is finished, you should see the following directory structure in the `results/MIIC/sam2_finetuning`:

```shell
results/MIIC/sam2_finetuning
├── epoch01_visualization/
...
├── best.pt
└── epoch1_iou0.xxx.pt
```

`epoch*_visualization` is the directory containing the visualization results of each epoch.
`best.pt` is the finetuned model weights with the highest validation mIoU. It is used as the default checkpoint for the mask refinement.

### 4.2 Pseudo-Labeling on Generated Data

In this section, we will perform the pseudo-labeling process for the generated data.

You can optionally disable features using the following flags:
- `--no_mask_refinement`: To skip mask refinement.
- `--no_caption`: To skip caption generation.

The workflow is as follows:

1. **Data Loading**: Loads the original images, original masks, generated images and a CSV file containing the generation details.
2. **Mask Clustering**: Clusters the masks using DBSCAN to group neighboring anomalies together.
3. **(Optional) Mask Refinement**: Refines the masks using the finetuned SAM2 model.
4. **Bbox and RLE Computation**: Computes bounding boxes and run-length encoding (RLE) for each clustered mask in COCO format.
5. **(Optional) Captioning**: Uses the Cosmos-Reason 1 7B model to generate captions for the generated images based on a provided prompt.
6. **Organization**: Organizes the outputs for downstream tasks.

Available arguments for the pseudo-labeling script are:

- `ori_image_dir`: The directory containing the original clean images used for generation.
- `gen_image_dir`: The directory containing the generated anomaly images.
- `mask_dir`: The directory containing the masks used for generation.
- `csv_path`: The path to the CSV file which is generated by PAIDF AnomalyGen.
- `output_dir`: The directory where the pseudo-labeled data will be saved.
- `no_mask_refinement`: If set, the mask refinement step will be skipped. This saves time if mask refinement is not needed or you don't have a finetuned SAM2 model.
- `no_caption`: If set, the captioning step will be skipped. This saves time if captions are not needed.
- `dbscan_eps`: DBSCAN eps parameter for clustering masks. Default is `0.2`.
- `dbscan_min_samples`: DBSCAN min_sample parameter for clustering masks. Default is `5`.
- `mask_refinement_checkpoint_path`: The path to the finetuned SAM2 model weights.
- `mask_refinement_strength`: The strength for the mask prompt in mask refinement. Set to `0.0` to disable this feature. Default is `1.0`.
- `mask_refinement_fallback_ratio`: The threshold ratio of the area for the refined mask that needs to be filtered. `0.5` means that if the area of  the refined mask is smaller than the area of the input mask by 50%, the refined mask will be filtered. Set to `0.0` to disable this feature. Default is `0.5`.
- `captioner_prompt_path`: The path to the caption prompt file used by the captioner. If not provided, a default prompt will be used. See `pseudo_label/default_caption_prompt.yaml` for the details.
- `captioner_num_gpus`: The number of GPUs to use for the captioning process. More GPUs can speed up the process. Default is `1`.
- `captioner_temperature`: Captioner temperature parameter for generating captions. Default is `0.01`.
- `captioner_max_tokens`: Captioner max_tokens parameter for generating captions. Default is `4096`.
- `captioner_seed`: Captioner seed parameter for generating captions. Default is `42`.

The supported image formats are `.jpg`, `.jpeg`, `.png`, `.bmp` and `.tiff`.

The mask values should be either 0 (background) or 255 (anomaly region). Specifically, the mask is binarized with a threshold of 127 (`binary_mask = (mask > 127)`).

**Note**:
When you run the pseudo-labeling process for the first time, the Cosmos-Reason model will be downloaded and saved to `./checkpoints`, just like the PAIDF AnomalyGen model. It might take some time depending on your network speed.

**Important**:
The quality of the generated captions may vary. We highly recommend creating a custom prompt that is tailored to your specific dataset and use case, as this can significantly improve the relevance and accuracy of the captions generated by the captioner. Replace the default prompt by setting `--captioner_prompt_path` to a custom prompt file.

In [ ]:
!conda run -n cosmos-predict2 \
    bash -c "python3 -m scripts.anomaly_gen.pseudo_label \
        --ori_image_dir=results/MIIC/example_output/original_image \
        --gen_image_dir=results/MIIC/example_output/reconstructed_image \
        --mask_dir=results/MIIC/example_output/original_mask \
        --csv_path=results/MIIC/example_output/SDG_result.csv \
        --output_dir=results/MIIC/pseudo_labeling \
        --mask_refinement_checkpoint_path=results/MIIC/sam2_finetuning/best.pt \
        --captioner_num_gpus=1"

You should see the following directory structure in the `results/MIIC/pseudo_labeling`:

```shell
results/MIIC/pseudo_labeling
├── captions
├── captions_with_meta
├── classification
├── images
├── masks
├── visualization
├── coco_annotations.json
└── ori_coco_annotations.json
```

For the TAO classification task, you can use the `classification` folder as the input. This folder contains the images organized by their corresponding labels, along with a `classes.txt` file that indicates the class names.

For the TAO detection tasks, you can use the `coco_annotations.json` and the `images` folder as the input. The `coco_annotations.json` contains both the bounding box and instance segmentation masks in COCO format. `ori_coco_annotations.json` contains the original annotations before mask refinement.

For the CLIP and VLM tasks, you can use the `captions` folder and the `images` folder as the input. The `captions` folder contains a text file for each image, with the generated caption inside.

We also provide a `visualization` folder that contains images with overlaid bboxes and masks for easy inspection of the generated data. Regarding `captions_with_meta`, you can find the captions along with their metadata (e.g., `image_type`, `anomaly_type`, `bboxes`, etc.).

## Congratulations!

You have successfully completed the tutorial.